# 分钟净执行价值：固定业界方法迁移实验

假设：首小时原始分钟路径在持仓和风险状态之外，能够预测延迟执行后到次日共同重置仓位的净调整收益。沿用原 f696/b472 模拟盘锚点、IBKR Tiered 和 10 万美元。固定两类 Ridge、三次年度训练；原账户与 SOXX 作对照。历史已复用，不能视为干净样本外。

方法选择见 [industry_selection.md](industry_selection.md)，冻结边界见 [plan.json](plan.json)。先修正草案中 60 分钟收益搭配次日费用的期限错误，再开始训练。

In [1]:
import os, sys, json
from pathlib import Path
root = Path.cwd()
while not (root / "SPEC.md").is_file(): root = root.parent
sys.path.insert(0, str(root))
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
from src.workflow.minute_action_value import prepare_minute_value, run_minute_value
card = root / "research/minute_net_action_value_20261013"
plan = json.loads((card / "plan.json").read_text())
prepared = prepare_minute_value(root=root, card=card, plan=plan)
prepared["audit"]["marks"]

{'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/minute_net_action_value_20261013',
 'market': 'all',
 'region': 'us',
 'freq': '1min',
 'as_of': '2026-10-09',
 'source': 'NAS observed minute opens/session-close execution point marks',
 'instruments': ['SOXL', 'SOXX'],
 'fields': ['$open',
  '$high',
  '$low',
  '$close',
  '$volume',
  '$execution_price',
  '$raw_execution_price'],
 'calendar_start': '2023-01-03 09:30:00',
 'calendar_end': '2026-10-09 16:00:00',
 'max_datetime': '2026-10-09 16:00:00',
 'row_count': 5676,
 'feature_file_count': 14,
 'audit_snapshot_hash': 'ba6e1e9df08c9ef9',
 'operation': 'qlib_store_build',
 'status': 'ok',
 'hash': 'e8377b0f28d5586c',
 'source_api': ['qlib.init',
  'qlib.data.storage.file_storage.FileFeatureStorage',
  'qlib.data.D.calendar',
  'qlib.data.D.instruments',
  'qlib.data.D.features'],
 'source_manifest_sha256': '18f1fb3f17907932ddd13881aea9199369181a92c9b6c77395d0b255100c68c3',
 'basis': {'basis': 'published_futu_affine_v1',
  'a

In [2]:
summary = run_minute_value(root=root, card=card, plan=plan, prepared=prepared)
import pandas as pd
display(pd.read_csv(card / "backtest_benchmark_comparison.csv")[["family", "slippage_bps", "return_pct", "max_drawdown_pct", "relative_wealth", "recovery_actions"]])
summary["gates"]

,family,slippage_bps,return_pct,max_drawdown_pct,relative_wealth,recovery_actions
0,current,0,233.080895,-26.668568,0.667682,0
1,state,0,235.512552,-26.667925,0.672556,26
2,minute,0,194.070528,-26.674987,0.589483,39
3,soxx,0,398.861540,-42.738804,1.000000,0
4,current,2,229.141547,-26.733271,0.659812,0
5,state,2,150.053954,-26.718700,0.501269,34
6,minute,2,189.748865,-26.726407,0.580843,39
7,soxx,2,398.841595,-42.742502,1.000000,0
8,current,5,194.983296,-26.801189,0.591372,0
9,state,5,200.677687,-26.818322,0.602788,29


{'net_soxx_excess_all_cost_scenarios': False,
 'max_drawdown_within_30pct_all_scenarios': True,
 'cvar_no_worse_than_soxx': True,
 'minute_increment_positive_all_scenarios': False,
 'annual_at_least_three_positive': False,
 'rolling_positive_at_least_70pct': False,
 'bootstrap_soxx_and_state_lower_bound_positive': False,
 'independent_forward_and_real_execution_qualified': False}

In [3]:
# 复用已完成的 Recorder；不新增训练、阈值搜索或回测。
import hashlib, numpy as np
from sklearn.metrics import roc_auc_score
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
names = ["inputs/frame", "analysis/teacher"] + [f"models/{family}_{year}" for family in ("state", "minute") for year in plan["fit_years"]]
objects, read_audit = load_verified_recorder_objects(card, expected_recorder_id=summary["recorder_id"], artifacts=names)
original_frame = objects["inputs/frame"]
minute_columns = [name for name in original_frame if name.startswith("minute_")] + ["feature_available_at"]
pd.testing.assert_frame_equal(original_frame[minute_columns], prepared["frame"][minute_columns])
guard = {"historical_feature_readback": "exactly_identical", "checked_sessions": len(original_frame),
         "invalid_or_missing_session_outputs": "remain_unknown; explicit price_valid flag and empty-hour guard added after run",
         "new_model_fits": 0, "new_backtests": 0, "executed_original_code_retained": True,
         "final_data_code_sha256": hashlib.sha256((root / "src/data/minute_action_marks.py").read_bytes()).hexdigest()}
teacher = objects["analysis/teacher"]
information = []
for year in plan["fit_years"]:
    evaluation = teacher.loc[teacher.index.year == year]
    for family in ("state", "minute"):
        model = objects[f"models/{family}_{year}"]
        estimate = model.predict_frame(evaluation)
        selected = estimate["mean"] - plan["confidence_z"] * estimate["mean_error"] > 0
        information.append({"year": year, "family": family, "reference_actions": len(evaluation),
            "training_actions": model.audit["observations"], "features": len(model.features),
            "rank_ic": evaluation.net_advantage.corr(estimate["mean"], method="spearman"),
            "positive_advantage_auc": roc_auc_score(evaluation.net_advantage.gt(0), estimate["mean"]),
            "rmse": float(np.sqrt(np.mean((evaluation.net_advantage - estimate["mean"]) ** 2))),
            "lcb_selected_reference_actions": int(selected.sum()),
            "lcb_selected_mean_advantage_bp": float(1e4 * evaluation.loc[selected, "net_advantage"].mean()),
            "unconditional_mean_advantage_bp": float(1e4 * evaluation.net_advantage.mean()),
            "role": "reused reference-state diagnostic; not own-policy realized returns or fresh holdout"})
information = pd.DataFrame(information)
information.to_csv(card / "information_metrics.csv", index=False)
summary["information_diagnosis"] = information.to_dict("records")
summary["final_data_guard_readback"] = guard
with R.uri_context(summary["recorder_uri"]):
    recorder = R.get_recorder(experiment_id=summary["experiment_id"], recorder_id=summary["recorder_id"])
    recorder.save_objects(artifact_path="sources", final_data_guard_code=(root / "src/data/minute_action_marks.py").read_text())
    recorder.save_objects(artifact_path="analysis", information_metrics=information, final_data_guard_readback=guard)
    recorder_summary = dict(summary)
    recorder_summary.pop("nas_evidence", None)
    recorder.save_objects(artifact_path="analysis", summary=recorder_summary)
    artifacts = Path(summary["recorder_local_dir"]) / "artifacts"
    recorder.log_metrics(registered_artifact_count=len([item for item in artifacts.rglob("*") if item.is_file()]))
entries = [{"path": str(item.relative_to(card)), "sha256": hashlib.sha256(item.read_bytes()).hexdigest(), "bytes": item.stat().st_size}
           for item in sorted(artifacts.rglob("*")) if item.is_file()]
atomic_json({"recorder_id": summary["recorder_id"], "entries": entries}, card / "verification_manifest.json")
summary["nas_evidence"] = archive_nas_recorder(card / "nas_binding.json", workspace_root=root, card=card,
    recorder_id=summary["recorder_id"], experiment_id=summary["experiment_id"])
atomic_json(summary, card / "summary.json")
from src.workflow.minute_action_value import render_conclusion
text = render_conclusion(summary, pd.read_csv(card / "backtest_benchmark_comparison.csv"))
text += "\n信息诊断：三次训练仅有 47 / 121 / 173 个成熟恢复机会，分钟模型共 29 个特征。2024/2025/2026 的净价值 rank IC 为 +0.088 / -0.077 / -0.128；2025/2026 被置信门槛选中的参考机会平均净价值为 -49.25 / -41.11 bp。2 bp 下分钟模型相对状态模型的区块置信界仍跨零，所以不能把局部收益差认作稳定分钟 alpha。\n"
text += "\n下一数据扩充要求：复用真实 2018 年以来分钟历史时，必须用当年之前冻结的分配/行动模型生成训练状态；不能让 2022 年最终模型反跑早期年份冒充当时可用模型。原始 2008 个交易日不等于 2008 个合格恢复标签。需先确认早期逐年模型与数据时点，保留坏行未知并保留压力日账户。这是下一信息资格工作，本轮不追加训练。\n"
(card / "conclusion.md").write_text(text)
display(information)
guard


,year,family,reference_actions,training_actions,features,rank_ic,positive_advantage_auc,rmse,lcb_selected_reference_actions,lcb_selected_mean_advantage_bp,unconditional_mean_advantage_bp,role
0,2024,state,74,47,17,-0.026760,0.485119,0.003740,27,1.527238,2.772207,reused reference-state diagnostic; not own-pol...
1,2024,minute,74,47,29,0.088012,0.533482,0.003947,29,3.927647,2.772207,reused reference-state diagnostic; not own-pol...
2,2025,state,52,121,17,-0.280799,0.430285,0.006500,3,-35.414220,-0.509291,reused reference-state diagnostic; not own-pol...
3,2025,minute,52,121,29,-0.077435,0.547226,0.006360,5,-49.252857,-0.509291,reused reference-state diagnostic; not own-pol...
4,2026,state,67,173,17,-0.141552,0.451906,0.004271,1,-38.554476,3.373419,reused reference-state diagnostic; not own-pol...
5,2026,minute,67,173,29,-0.128183,0.492740,0.004784,1,-41.108016,3.373419,reused reference-state diagnostic; not own-pol...


{'historical_feature_readback': 'exactly_identical',
 'checked_sessions': 944,
 'invalid_or_missing_session_outputs': 'remain_unknown; explicit price_valid flag and empty-hour guard added after run',
 'new_model_fits': 0,
 'new_backtests': 0,
 'executed_original_code_retained': True,
 'final_data_code_sha256': '1b8a5756d65c3ac3183eafe5209945e33e1cae725bee3abf82347a304b4e6104'}

In [4]:
from IPython.display import Markdown, display
display(pd.read_csv(card / "regime_metrics.csv")[["year", "family", "slippage_bps", "return_pct", "relative_wealth", "max_drawdown_pct"]])
display(Markdown((card / "conclusion.md").read_text()))

,year,family,slippage_bps,return_pct,relative_wealth,max_drawdown_pct
0,2023,current,0,27.758547,0.815486,-20.293019
1,2024,current,0,10.603636,0.915980,-16.714603
2,2025,current,0,32.528226,0.895995,-14.752368
3,2026,current,0,77.861509,0.997612,-26.668568
4,2023,state,0,27.758547,0.815486,-20.293019
5,2024,state,0,11.978704,0.927368,-18.078708
6,2025,state,0,32.338537,0.894713,-14.746658
7,2026,state,0,77.213599,0.993978,-26.667925
8,2023,minute,0,27.758547,0.815486,-20.293019
9,2024,minute,0,7.967631,0.894150,-17.735481


# 分钟净执行价值实验结论

已执行固定的 6 次 Qlib 年度滚动训练和 12 个独立扣费账户；数据固定到 2026-10-09，覆盖 944 个持有区间。锚点为原模拟盘 f696/b472，未改模拟盘。

已知信息截至 10:30，假定 60 秒后可用，10:31 决策并延迟到 10:32 实际分钟开盘成交。标签覆盖到次日 09:30，并对 HOLD/调整分支计入共同重置仓位的费用；未来重置目标只用于标签。

|策略|滑点 bp|累计净收益 %|最大回撤 %|恢复次数|
|---|---:|---:|---:|---:|
|current|0|233.08|-26.67|0|
|state|0|235.51|-26.67|26|
|minute|0|194.07|-26.67|39|
|soxx|0|398.86|-42.74|0|
|current|2|229.14|-26.73|0|
|state|2|150.05|-26.72|34|
|minute|2|189.75|-26.73|39|
|soxx|2|398.84|-42.74|0|
|current|5|194.98|-26.80|0|
|state|5|200.68|-26.82|29|
|minute|5|183.97|-26.80|38|
|soxx|5|398.81|-42.75|0|

决定：`not_promoted_minute_net_value_gates_failed`。门槛：{"net_soxx_excess_all_cost_scenarios": false, "max_drawdown_within_30pct_all_scenarios": true, "cvar_no_worse_than_soxx": true, "minute_increment_positive_all_scenarios": false, "annual_at_least_three_positive": false, "rolling_positive_at_least_70pct": false, "bootstrap_soxx_and_state_lower_bound_positive": false, "independent_forward_and_real_execution_qualified": false}。分钟相对状态模型的累计收益百分点差：{'0': -41.44202410286735, '2': 39.69491033791792, '5': -16.70705668517718}。年度和滚动差异见 regime_metrics.csv、window_metrics.csv。

风险：历史已重复研究，没有干净样本外或合格前向证据；条件均值 HAC 误差不是未来收益覆盖保证；标签来自固定参考账户，未模拟完整未来风控树。当前 IBKR Tiered 费表为历史投影；滑点、通道、报价时点为假设，复权 NAV 未认证实际股息/成交对账。

本固定路线结束，不反转信号或追加参数搜索。下一信息需求：若要评估交易可实现性，需真实首收时间、买卖报价、成交和佣金回执；若要继续寻找预测信息，需独立机制和足够的成熟恢复样本，不能从本窗口再挑阈值。Notebook、summary 与 NAS Recorder 是审计入口。

信息诊断：三次训练仅有 47 / 121 / 173 个成熟恢复机会，分钟模型共 29 个特征。2024/2025/2026 的净价值 rank IC 为 +0.088 / -0.077 / -0.128；2025/2026 被置信门槛选中的参考机会平均净价值为 -49.25 / -41.11 bp。2 bp 下分钟模型相对状态模型的区块置信界仍跨零，所以不能把局部收益差认作稳定分钟 alpha。

下一数据扩充要求：复用真实 2018 年以来分钟历史时，必须用当年之前冻结的分配/行动模型生成训练状态；不能让 2022 年最终模型反跑早期年份冒充当时可用模型。原始 2008 个交易日不等于 2008 个合格恢复标签。需先确认早期逐年模型与数据时点，保留坏行未知并保留压力日账户。这是下一信息资格工作，本轮不追加训练。


# 分钟净执行价值实验结论

已执行固定的 6 次 Qlib 年度滚动训练和 12 个独立扣费账户；数据固定到 2026-10-09，覆盖 944 个持有区间。锚点为原模拟盘 f696/b472，未改模拟盘。

已知信息截至 10:30，假定 60 秒后可用，10:31 决策并延迟到 10:32 实际分钟开盘成交。标签覆盖到次日 09:30，并对 HOLD/调整分支计入共同重置仓位的费用；未来重置目标只用于标签。

|策略|滑点 bp|累计净收益 %|最大回撤 %|恢复次数|
|---|---:|---:|---:|---:|
|current|0|233.08|-26.67|0|
|state|0|235.51|-26.67|26|
|minute|0|194.07|-26.67|39|
|soxx|0|398.86|-42.74|0|
|current|2|229.14|-26.73|0|
|state|2|150.05|-26.72|34|
|minute|2|189.75|-26.73|39|
|soxx|2|398.84|-42.74|0|
|current|5|194.98|-26.80|0|
|state|5|200.68|-26.82|29|
|minute|5|183.97|-26.80|38|
|soxx|5|398.81|-42.75|0|

决定：`not_promoted_minute_net_value_gates_failed`。门槛：{"net_soxx_excess_all_cost_scenarios": false, "max_drawdown_within_30pct_all_scenarios": true, "cvar_no_worse_than_soxx": true, "minute_increment_positive_all_scenarios": false, "annual_at_least_three_positive": false, "rolling_positive_at_least_70pct": false, "bootstrap_soxx_and_state_lower_bound_positive": false, "independent_forward_and_real_execution_qualified": false}。分钟相对状态模型的累计收益百分点差：{'0': -41.44202410286735, '2': 39.69491033791792, '5': -16.70705668517718}。年度和滚动差异见 regime_metrics.csv、window_metrics.csv。

风险：历史已重复研究，没有干净样本外或合格前向证据；条件均值 HAC 误差不是未来收益覆盖保证；标签来自固定参考账户，未模拟完整未来风控树。当前 IBKR Tiered 费表为历史投影；滑点、通道、报价时点为假设，复权 NAV 未认证实际股息/成交对账。

本固定路线结束，不反转信号或追加参数搜索。下一信息需求：若要评估交易可实现性，需真实首收时间、买卖报价、成交和佣金回执；若要继续寻找预测信息，需独立机制和足够的成熟恢复样本，不能从本窗口再挑阈值。Notebook、summary 与 NAS Recorder 是审计入口。

信息诊断：三次训练仅有 47 / 121 / 173 个成熟恢复机会，分钟模型共 29 个特征。2024/2025/2026 的净价值 rank IC 为 +0.088 / -0.077 / -0.128；2025/2026 被置信门槛选中的参考机会平均净价值为 -49.25 / -41.11 bp。2 bp 下分钟模型相对状态模型的区块置信界仍跨零，所以不能把局部收益差认作稳定分钟 alpha。

下一数据扩充要求：复用真实 2018 年以来分钟历史时，必须用当年之前冻结的分配/行动模型生成训练状态；不能让 2022 年最终模型反跑早期年份冒充当时可用模型。原始 2008 个交易日不等于 2008 个合格恢复标签。需先确认早期逐年模型与数据时点，保留坏行未知并保留压力日账户。这是下一信息资格工作，本轮不追加训练。
